# 04_gold — Silver → Gold (Star Schema)

Builds the business-ready Gold layer from `silver_listings`.

| Table | Type | Grain | Load strategy |
|---|---|---|---|
| `gold_dim_date` | Dimension | 1 row per calendar day (+ `-1` Unknown) | MERGE on `date_key` |
| `gold_dim_category` | Dimension | 1 row per parent category + subcategory | MERGE on `category_key` |
| `gold_dim_condition` | Dimension | 1 row per item condition | MERGE on `condition_key` |
| `gold_dim_seller` | Dimension | 1 row per hashed seller | MERGE on `seller_key` |
| `gold_fact_listings` | Fact | 1 row per listing (`item_id`) | Incremental MERGE (watermark on Silver `load_timestamp`) |
| `gold_agg_daily_category` | Aggregate fact | day × category × condition | Full overwrite (fully derived from the fact) |

Every table load is logged to `pipeline_execution_logs` with layer `Silver-to-Gold`, rows inserted / updated / aggregated and execution time.

In [0]:
%run ../00_config/00_config

## 1. Imports, parameters and table names

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime, date
import uuid

# Widget: set to "true" to rebuild the whole fact from Silver instead of only new/changed rows
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "Full Refresh Gold Fact")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"

SCHEMA = "ebay_pipeline"
TABLE_SILVER_LISTINGS = f"{SCHEMA}.silver_listings"
TABLE_LOGS            = f"{SCHEMA}.pipeline_execution_logs"

DIM_DATE      = f"{SCHEMA}.gold_dim_date"
DIM_CATEGORY  = f"{SCHEMA}.gold_dim_category"
DIM_CONDITION = f"{SCHEMA}.gold_dim_condition"
DIM_SELLER    = f"{SCHEMA}.gold_dim_seller"
FACT_LISTINGS = f"{SCHEMA}.gold_fact_listings"
AGG_DAILY_CAT = f"{SCHEMA}.gold_agg_daily_category"

LAYER = "Silver-to-Gold"
UNKNOWN_DATE_KEY = -1

print(f"Gold run started | full_refresh={FULL_REFRESH}")

## 2. Logging (continues the Phase 2 audit table)

Adds a `rows_aggregated` column to `pipeline_execution_logs` (one-time, idempotent) so Gold runs can record how many Silver/fact rows were aggregated. Older Bronze/Silver rows simply have `NULL` there.

In [0]:
log_cols = [f.name for f in spark.table(TABLE_LOGS).schema.fields]
if "rows_aggregated" not in log_cols:
    spark.sql(f"ALTER TABLE {TABLE_LOGS} ADD COLUMNS (rows_aggregated BIGINT)")
    print("Added rows_aggregated column to pipeline_execution_logs")

gold_log_schema = StructType([
    StructField("log_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("parameter_processed", StringType(), True),
    StructField("execution_start_time", TimestampType(), False),
    StructField("execution_end_time", TimestampType(), True),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("rows_aggregated", LongType(), True),
])

def log_gold(table_name, param, start, status, inserted=0, updated=0, aggregated=0, error=None):
    row = [(
        str(uuid.uuid4()), LAYER, table_name, str(param), start, datetime.now(), status,
        int(inserted), int(updated), (str(error)[:4000] if error else None), int(aggregated)
    )]
    spark.createDataFrame(row, gold_log_schema).write.format("delta").mode("append").saveAsTable(TABLE_LOGS)

def table_version(table_name):
    return spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").collect()[0]["version"]

def merge_metrics(table_name, version_before, merge_result):
    """
    Exact inserted/updated counts of the MERGE just run.
    1) Databricks returns a metrics DataFrame from execute() - use it when present.
    2) Otherwise read the Delta transaction log, but only if the MERGE created a new
       version (a no-op MERGE must report 0, not the previous run's numbers).
    """
    if merge_result is not None:
        r = merge_result.collect()[0].asDict()
        if "num_inserted_rows" in r:
            return int(r["num_inserted_rows"]), int(r["num_updated_rows"])
    h = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").collect()[0]
    if h["version"] == version_before or h["operation"] != "MERGE":
        return 0, 0
    m = h["operationMetrics"]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))

def run_gold_step(table_name, param, step_fn):
    """Runs one Gold load, logs SUCCESS/FAILURE with metrics, and re-raises on failure so the Job fails."""
    start = datetime.now()
    try:
        inserted, updated, aggregated = step_fn()
        log_gold(table_name, param, start, "SUCCESS", inserted, updated, aggregated)
        secs = (datetime.now() - start).total_seconds()
        print(f"[{LAYER}] {table_name}: inserted={inserted}, updated={updated}, "
              f"aggregated={aggregated} | {secs:.1f}s")
    except Exception as e:
        log_gold(table_name, param, start, "FAILURE", error=e)
        raise

def merge_upsert(target, df_src, key_col, ignore_in_change_check=("gold_load_timestamp",)):
    """
    Idempotent SCD Type 1 upsert.
    Only updates a row when a business column actually changed, so re-running
    the same data gives rows_updated = 0 (proves idempotency in the logs).
    """
    compare_cols = [c for c in df_src.columns if c != key_col and c not in ignore_in_change_check]
    change_cond = " OR ".join([f"NOT (t.`{c}` <=> s.`{c}`)" for c in compare_cols])
    version_before = table_version(target)
    result = (
        DeltaTable.forName(spark, target).alias("t")
        .merge(df_src.alias("s"), f"t.`{key_col}` = s.`{key_col}`")
        .whenMatchedUpdate(condition=change_cond, set={c: f"s.`{c}`" for c in df_src.columns})
        .whenNotMatchedInsertAll()
        .execute()
    )
    return merge_metrics(target, version_before, result)

## 3. Create Gold tables (DDL)

- Surrogate keys are **deterministic hashes** (`xxhash64`) of the natural key, so the same input always produces the same key and re-runs stay idempotent (identity columns would not).
- **Informational PRIMARY / FOREIGN KEY constraints** (Unity Catalog) document the star schema; Power BI can read them to suggest relationships.
- **Liquid Clustering** on the fact (`date_key`, `category_key`) and the aggregate (`date_key`) — the columns dashboards filter on most. Dimensions are tiny, so they are not clustered.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_DATE} (
    date_key      INT NOT NULL COMMENT 'yyyyMMdd; -1 = Unknown',
    full_date     DATE,
    day_of_month  INT,
    day_name      STRING,
    day_of_week   INT COMMENT '1 = Sunday ... 7 = Saturday',
    is_weekend    BOOLEAN,
    week_of_year  INT,
    month_num     INT,
    month_name    STRING,
    year_month    STRING,
    quarter       INT,
    year          INT,
    CONSTRAINT pk_gold_dim_date PRIMARY KEY (date_key)
) USING DELTA
COMMENT 'Gold calendar dimension'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_CATEGORY} (
    category_key        BIGINT NOT NULL COMMENT 'xxhash64(parent_category, subcategory)',
    parent_category     STRING,
    subcategory         STRING,
    gold_load_timestamp TIMESTAMP,
    CONSTRAINT pk_gold_dim_category PRIMARY KEY (category_key)
) USING DELTA
COMMENT 'Gold category dimension (conformed parent/subcategory names)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_CONDITION} (
    condition_key       BIGINT NOT NULL COMMENT 'xxhash64(lower(condition))',
    condition           STRING,
    condition_group     STRING COMMENT 'New / Used / Open Box / Refurbished / For Parts / Other / Unknown',
    gold_load_timestamp TIMESTAMP,
    CONSTRAINT pk_gold_dim_condition PRIMARY KEY (condition_key)
) USING DELTA
COMMENT 'Gold item condition dimension'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM_SELLER} (
    seller_key          BIGINT NOT NULL COMMENT 'xxhash64(seller_id_hashed)',
    seller_id_hashed    STRING COMMENT 'Salted SHA-256 pseudonym from Silver - no raw PII in Gold',
    first_listing_date  DATE,
    last_listing_date   DATE,
    total_listings      BIGINT,
    gold_load_timestamp TIMESTAMP,
    CONSTRAINT pk_gold_dim_seller PRIMARY KEY (seller_key)
) USING DELTA
COMMENT 'Gold seller dimension (pseudonymised)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FACT_LISTINGS} (
    item_id                 STRING NOT NULL COMMENT 'eBay item id (degenerate dimension)',
    date_key                INT,
    category_key            BIGINT,
    condition_key           BIGINT,
    seller_key              BIGINT,
    price                   DOUBLE,
    currency                STRING,
    price_band              STRING,
    item_creation_timestamp TIMESTAMP,
    batch_id                STRING,
    silver_load_timestamp   TIMESTAMP COMMENT 'Watermark column for incremental Gold loads',
    gold_load_timestamp     TIMESTAMP,
    CONSTRAINT pk_gold_fact_listings PRIMARY KEY (item_id),
    CONSTRAINT fk_fact_date      FOREIGN KEY (date_key)      REFERENCES {DIM_DATE},
    CONSTRAINT fk_fact_category  FOREIGN KEY (category_key)  REFERENCES {DIM_CATEGORY},
    CONSTRAINT fk_fact_condition FOREIGN KEY (condition_key) REFERENCES {DIM_CONDITION},
    CONSTRAINT fk_fact_seller    FOREIGN KEY (seller_key)    REFERENCES {DIM_SELLER}
) USING DELTA
CLUSTER BY (date_key, category_key)
COMMENT 'Gold listing fact - one row per eBay listing'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AGG_DAILY_CAT} (
    date_key            INT,
    category_key        BIGINT,
    condition_key       BIGINT,
    listing_count       BIGINT,
    distinct_sellers    BIGINT,
    total_listed_value  DOUBLE,
    avg_price           DOUBLE,
    median_price        DOUBLE,
    min_price           DOUBLE,
    max_price           DOUBLE,
    gold_load_timestamp TIMESTAMP
) USING DELTA
CLUSTER BY (date_key)
COMMENT 'Gold daily aggregate by category and condition - feeds dashboard trend visuals'
""")

print("Gold tables ready")

## 4. Read Silver and conform attributes

- Blank / NULL attributes become an explicit **Unknown** member instead of breaking joins.
- Category names are **conformed**: the source mixes spellings (e.g. `jewelery` / `jewelry`, `ebay motor`), so they are normalised before keys are hashed.

In [0]:
def clean(col_name, default):
    c = F.trim(F.col(col_name))
    return F.when(c.isNull() | (c == ""), F.lit(default)).otherwise(c)

def normalise_category_text(c):
    c = F.lower(c)
    c = F.regexp_replace(c, r"jewelery", "jewelry")
    c = F.regexp_replace(c, r"land read estate", "land real estate")
    c = F.regexp_replace(c, r"^ebay motor$", "ebay motors")
    c = F.regexp_replace(c, r"\s+", " ")
    return F.initcap(F.trim(c))

df_silver = spark.table(TABLE_SILVER_LISTINGS)

df_base = (
    df_silver
    .select(
        F.col("item_id"),
        F.col("price"),
        F.col("currency"),
        F.col("item_creation_timestamp"),
        F.col("batch_id"),
        clean("subcategory", "unknown").alias("subcategory_raw"),
        clean("condition", "Unknown").alias("condition"),
        clean("seller_id_hashed", "UNKNOWN").alias("seller_id_hashed"),
        F.col("load_timestamp").alias("silver_load_timestamp"),
    )
    # 'parent-sub' -> parent_category / subcategory (split on the FIRST hyphen only)
    .withColumn("has_split", F.col("subcategory_raw").contains("-"))
    .withColumn("parent_category", normalise_category_text(
        F.when(F.col("has_split"), F.regexp_extract("subcategory_raw", r"^([^-]+)-", 1))
         .otherwise(F.col("subcategory_raw"))))
    .withColumn("subcategory", normalise_category_text(
        F.when(F.col("has_split"), F.regexp_extract("subcategory_raw", r"^[^-]+-(.*)$", 1))
         .otherwise(F.col("subcategory_raw"))))
    .drop("has_split", "subcategory_raw")
    # Deterministic surrogate keys
    .withColumn("category_key",  F.xxhash64("parent_category", "subcategory"))
    .withColumn("condition_key", F.xxhash64(F.lower("condition")))
    .withColumn("seller_key",    F.xxhash64("seller_id_hashed"))
    .withColumn("date_key", F.coalesce(
        F.date_format("item_creation_timestamp", "yyyyMMdd").cast("int"),
        F.lit(UNKNOWN_DATE_KEY)))
)

silver_count = df_silver.count()
print(f"Silver rows available: {silver_count}")

## 5. Load dimensions

In [0]:
# ---------- dim_date ----------
def load_dim_date():
    bounds = df_base.agg(
        F.min(F.to_date("item_creation_timestamp")).alias("mn"),
        F.max(F.to_date("item_creation_timestamp")).alias("mx"),
    ).collect()[0]
    start_d = min(bounds["mn"] or date(2020, 1, 1), date(2020, 1, 1))
    end_d   = max(bounds["mx"] or date(2030, 12, 31), date(2030, 12, 31))

    df_dates = spark.sql(f"""
        SELECT
            CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
            d                                       AS full_date,
            day(d)                                  AS day_of_month,
            date_format(d, 'EEEE')                  AS day_name,
            dayofweek(d)                            AS day_of_week,
            dayofweek(d) IN (1, 7)                  AS is_weekend,
            weekofyear(d)                           AS week_of_year,
            month(d)                                AS month_num,
            date_format(d, 'MMMM')                  AS month_name,
            date_format(d, 'yyyy-MM')               AS year_month,
            quarter(d)                              AS quarter,
            year(d)                                 AS year
        FROM (SELECT explode(sequence(DATE'{start_d}', DATE'{end_d}', INTERVAL 1 DAY)) AS d)
        UNION ALL
        SELECT -1, CAST(NULL AS DATE), CAST(NULL AS INT), 'Unknown', CAST(NULL AS INT),
               CAST(NULL AS BOOLEAN), CAST(NULL AS INT), CAST(NULL AS INT), 'Unknown', 'Unknown',
               CAST(NULL AS INT), CAST(NULL AS INT)
    """)
    ins, upd = merge_upsert(DIM_DATE, df_dates, "date_key", ignore_in_change_check=())
    return ins, upd, 0

run_gold_step(DIM_DATE, "calendar 2020-01-01..2030-12-31 (extends to data range)", load_dim_date)

# ---------- dim_category ----------
def load_dim_category():
    df = (df_base.select("category_key", "parent_category", "subcategory").distinct()
          .withColumn("gold_load_timestamp", F.current_timestamp()))
    ins, upd = merge_upsert(DIM_CATEGORY, df, "category_key")
    return ins, upd, silver_count

run_gold_step(DIM_CATEGORY, "distinct subcategory from silver_listings", load_dim_category)

# ---------- dim_condition ----------
def load_dim_condition():
    c = F.lower(F.col("condition"))
    df = (
        df_base.select("condition_key", "condition").dropDuplicates(["condition_key"])
        .withColumn("condition_group",
            F.when(c == "unknown", "Unknown")
             .when(c.contains("open box"), "Open Box")
             .when(c.contains("refurbished"), "Refurbished")
             .when(c.contains("parts") | c.contains("not working"), "For Parts")
             .when(c.startswith("new"), "New")
             .when(c.contains("used") | c.contains("pre-owned"), "Used")
             .otherwise("Other"))
        .withColumn("gold_load_timestamp", F.current_timestamp())
    )
    ins, upd = merge_upsert(DIM_CONDITION, df, "condition_key")
    return ins, upd, silver_count

run_gold_step(DIM_CONDITION, "distinct condition from silver_listings", load_dim_condition)

# ---------- dim_seller ----------
def load_dim_seller():
    df = (
        df_base.groupBy("seller_key", "seller_id_hashed")
        .agg(
            F.min(F.to_date("item_creation_timestamp")).alias("first_listing_date"),
            F.max(F.to_date("item_creation_timestamp")).alias("last_listing_date"),
            F.count("*").alias("total_listings"),
        )
        .withColumn("gold_load_timestamp", F.current_timestamp())
    )
    ins, upd = merge_upsert(DIM_SELLER, df, "seller_key")
    return ins, upd, silver_count

run_gold_step(DIM_SELLER, "seller_id_hashed grouped from silver_listings", load_dim_seller)

## 6. Load the fact table (incremental)

Watermark = latest Silver `load_timestamp` already in the fact. Only Silver rows newer than that are processed (Silver's MERGE refreshes `load_timestamp` on every insert/update, so changed listings are picked up too). `full_refresh = true` reprocesses everything.

In [0]:
def load_fact_listings():
    watermark = None
    if not FULL_REFRESH:
        watermark = spark.table(FACT_LISTINGS).agg(F.max("silver_load_timestamp")).collect()[0][0]

    df_changed = df_base if watermark is None else df_base.filter(F.col("silver_load_timestamp") > F.lit(watermark))
    rows_read = df_changed.count()
    if rows_read == 0:
        print("No new Silver rows since last Gold run - fact unchanged")
        return 0, 0, 0

    p = F.col("price")
    df_fact = (
        df_changed
        .withColumn("price_band",
            F.when(p < 25, "1. Under $25")
             .when(p < 100, "2. $25-100")
             .when(p < 500, "3. $100-500")
             .when(p < 2000, "4. $500-2K")
             .otherwise("5. $2K+"))
        .withColumn("gold_load_timestamp", F.current_timestamp())
        .select("item_id", "date_key", "category_key", "condition_key", "seller_key",
                "price", "currency", "price_band", "item_creation_timestamp", "batch_id",
                "silver_load_timestamp", "gold_load_timestamp")
        .dropDuplicates(["item_id"])
    )
    ins, upd = merge_upsert(FACT_LISTINGS, df_fact, "item_id",
                            ignore_in_change_check=("gold_load_timestamp", "silver_load_timestamp"))
    return ins, upd, rows_read

fact_param = "FULL_REFRESH" if FULL_REFRESH else "silver_load_timestamp > last fact watermark"
run_gold_step(FACT_LISTINGS, fact_param, load_fact_listings)

## 7. Load the daily aggregate (full overwrite)

Fully derived from the fact and small, so a deterministic overwrite is simpler and always correct (no risk of double counting). `INSERT OVERWRITE` keeps the table definition and its Liquid Clustering.

**For the report:** this GROUP BY is the heaviest step (shuffle / Exchange). Open *See performance → query profile* on this cell to capture the execution DAG.

In [0]:
def load_agg_daily_category():
    df_fact = spark.table(FACT_LISTINGS)
    rows_aggregated = df_fact.count()

    df_agg = (
        df_fact.groupBy("date_key", "category_key", "condition_key")
        .agg(
            F.count("*").alias("listing_count"),
            F.countDistinct("seller_key").alias("distinct_sellers"),
            F.round(F.sum("price"), 2).alias("total_listed_value"),
            F.round(F.avg("price"), 2).alias("avg_price"),
            F.round(F.percentile_approx("price", 0.5), 2).alias("median_price"),
            F.min("price").alias("min_price"),
            F.max("price").alias("max_price"),
        )
        .withColumn("gold_load_timestamp", F.current_timestamp())
    )
    target_cols = spark.table(AGG_DAILY_CAT).columns
    df_agg.select(*target_cols).write.insertInto(AGG_DAILY_CAT, overwrite=True)

    rows_written = spark.table(AGG_DAILY_CAT).count()
    return rows_written, 0, rows_aggregated

run_gold_step(AGG_DAILY_CAT, "full overwrite from gold_fact_listings", load_agg_daily_category)

## 8. Read optimisation — OPTIMIZE + Liquid Clustering

Compacts small files and clusters data by the clustering keys. On this small dataset the speed gain is negligible; the point is that the same commands keep reads fast at enterprise scale. Compare `numFiles` before/after for the report.

In [0]:
for t in [FACT_LISTINGS, AGG_DAILY_CAT]:
    before = spark.sql(f"DESCRIBE DETAIL {t}").select("numFiles", "sizeInBytes", "clusteringColumns").collect()[0]
    spark.sql(f"OPTIMIZE {t}")
    after = spark.sql(f"DESCRIBE DETAIL {t}").select("numFiles", "sizeInBytes").collect()[0]
    print(f"{t}: clustering={before['clusteringColumns']} | files {before['numFiles']} -> {after['numFiles']} "
          f"| bytes {before['sizeInBytes']} -> {after['sizeInBytes']}")

## 9. Validation checks 

In [0]:
checks = {
    "fact PK duplicates (expect 0)":
        f"SELECT COUNT(*) FROM (SELECT item_id FROM {FACT_LISTINGS} GROUP BY item_id HAVING COUNT(*) > 1)",
    "orphan date_key (expect 0)":
        f"SELECT COUNT(*) FROM {FACT_LISTINGS} f LEFT ANTI JOIN {DIM_DATE} d ON f.date_key = d.date_key",
    "orphan category_key (expect 0)":
        f"SELECT COUNT(*) FROM {FACT_LISTINGS} f LEFT ANTI JOIN {DIM_CATEGORY} d ON f.category_key = d.category_key",
    "orphan condition_key (expect 0)":
        f"SELECT COUNT(*) FROM {FACT_LISTINGS} f LEFT ANTI JOIN {DIM_CONDITION} d ON f.condition_key = d.condition_key",
    "orphan seller_key (expect 0)":
        f"SELECT COUNT(*) FROM {FACT_LISTINGS} f LEFT ANTI JOIN {DIM_SELLER} d ON f.seller_key = d.seller_key",
    "silver rows minus fact rows (expect 0)":
        f"SELECT (SELECT COUNT(*) FROM {TABLE_SILVER_LISTINGS}) - (SELECT COUNT(*) FROM {FACT_LISTINGS})",
    "agg listing_count minus fact rows (expect 0)":
        f"SELECT (SELECT SUM(listing_count) FROM {AGG_DAILY_CAT}) - (SELECT COUNT(*) FROM {FACT_LISTINGS})",
}

failed = []
for name, q in checks.items():
    val = spark.sql(q).collect()[0][0]
    status = "OK" if val == 0 else "CHECK"
    if val != 0:
        failed.append(name)
    print(f"[{status}] {name}: {val}")

assert not failed, f"Gold validation failed: {failed}"
print("All Gold validation checks passed")

In [0]:
%sql
-- Gold step telemetry: rows and execution time per table
SELECT
    table_name,
    status,
    parameter_processed,
    rows_inserted,
    rows_updated,
    rows_aggregated,
    ROUND(unix_timestamp(execution_end_time) - unix_timestamp(execution_start_time), 1) AS duration_sec,
    execution_start_time
FROM ebay_pipeline.pipeline_execution_logs
WHERE layer = 'Silver-to-Gold'
ORDER BY execution_start_time DESC
LIMIT 20;

## 10. Sample business queries on the star schema

In [0]:
%sql
-- Q1: Highest-value markets - average / median price by category
SELECT c.parent_category, c.subcategory,
       COUNT(*)                         AS listings,
       ROUND(AVG(f.price), 2)           AS avg_price,
       ROUND(percentile_approx(f.price, 0.5), 2) AS median_price
FROM ebay_pipeline.gold_fact_listings f
JOIN ebay_pipeline.gold_dim_category c ON f.category_key = c.category_key
GROUP BY c.parent_category, c.subcategory
ORDER BY median_price DESC;

In [0]:
%sql
-- Q3: New vs Used price gap per category
SELECT c.parent_category,
       ROUND(AVG(CASE WHEN d.condition_group = 'New'  THEN f.price END), 2) AS avg_new_price,
       ROUND(AVG(CASE WHEN d.condition_group = 'Used' THEN f.price END), 2) AS avg_used_price,
       ROUND(AVG(CASE WHEN d.condition_group = 'New'  THEN f.price END)
           - AVG(CASE WHEN d.condition_group = 'Used' THEN f.price END), 2) AS new_premium
FROM ebay_pipeline.gold_fact_listings f
JOIN ebay_pipeline.gold_dim_category  c ON f.category_key  = c.category_key
JOIN ebay_pipeline.gold_dim_condition d ON f.condition_key = d.condition_key
GROUP BY c.parent_category
ORDER BY new_premium DESC;

In [0]:
%sql
-- Q5: Seller concentration - share of listings held by the top 10 sellers per category
WITH seller_counts AS (
    SELECT c.parent_category, f.seller_key, COUNT(*) AS listings,
           ROW_NUMBER() OVER (PARTITION BY c.parent_category ORDER BY COUNT(*) DESC) AS rnk
    FROM ebay_pipeline.gold_fact_listings f
    JOIN ebay_pipeline.gold_dim_category c ON f.category_key = c.category_key
    GROUP BY c.parent_category, f.seller_key
)
SELECT parent_category,
       SUM(listings)                                              AS total_listings,
       SUM(CASE WHEN rnk <= 10 THEN listings ELSE 0 END)          AS top10_listings,
       ROUND(100 * SUM(CASE WHEN rnk <= 10 THEN listings ELSE 0 END) / SUM(listings), 1) AS top10_share_pct
FROM seller_counts
GROUP BY parent_category
ORDER BY top10_share_pct DESC;

In [0]:
   %sql
   SELECT layer, table_name, status, rows_inserted, rows_updated, rows_aggregated,
          ROUND(unix_timestamp(execution_end_time) - unix_timestamp(execution_start_time), 1) AS duration_sec,
          execution_start_time
   FROM ebay_pipeline.pipeline_execution_logs
   ORDER BY execution_start_time DESC
   LIMIT 12;